# 117 — Delta-ML with All Fingerprint Types

Extends nb111 (ECFP4+ECFP6+MACCS) with additional fingerprint types:
- Atom-pair fingerprints (2048 bits)
- Topological torsion fingerprints (2048 bits)
- RDKit topological fingerprints (2048 bits)

Also uses more estimators for delta LGBM (1200 vs 800 in nb111).

Goal: beat nb111 OOF RAE = 0.2480.

In [1]:
import os, sys, warnings
os.environ["PYTHONIOENCODING"] = "utf-8"
if hasattr(sys.stdout, "reconfigure"): sys.stdout.reconfigure(encoding="utf-8")
sys.path.insert(0, "../src")
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy import stats
from pathlib import Path
from rdkit import Chem
from rdkit.Chem import AllChem, MACCSkeys, rdMolDescriptors
from pxr.data import load_train, load_test
from pxr.featurize import combined, impute
from pxr.eval import rae, scaffold_kfold_indices
from pxr.chem import bemis_murcko, morgan_fp_batch, compute_physchem
from pxr.paths import DATA_PROCESSED, SUBMISSIONS
SEED = 42; N_FOLDS = 5
LGBM_BASE = dict(n_estimators=1200, num_leaves=64, learning_rate=0.04,
                 min_child_samples=10, subsample=0.8, colsample_bytree=0.8,
                 reg_alpha=0.1, reg_lambda=0.1, random_state=SEED, verbose=-1, n_jobs=4)
print("imports OK")

imports OK


In [2]:
def full_metrics(y_true, y_pred, label=""):
    yt = np.asarray(y_true, float); yp = np.asarray(y_pred, float)
    msk = np.isfinite(yt) & np.isfinite(yp); yt, yp = yt[msk], yp[msk]
    mae = float(np.mean(np.abs(yt-yp)))
    rae_v = mae / float(np.mean(np.abs(yt-yt.mean()))) if yt.std()>0 else float("nan")
    r2  = 1-np.sum((yt-yp)**2)/np.sum((yt-yt.mean())**2) if yt.std()>0 else float("nan")
    pr, _ = stats.pearsonr(yt, yp); sp, _ = stats.spearmanr(yt, yp)
    kt, _ = stats.kendalltau(yt, yp)
    m = dict(RAE=rae_v, MAE=mae, R2=float(r2), Pearson=float(pr), Spearman=float(sp), Kendall=float(kt))
    if label:
        print(f"  [{label}] RAE={rae_v:.4f} MAE={mae:.4f} R2={r2:.4f} r={pr:.4f} rho={sp:.4f}")
    return m

In [3]:
# --- All fingerprint batch functions ---
def ecfp4_batch(smiles_list, n_bits=2048):
    fps = []
    for s in smiles_list:
        mol = Chem.MolFromSmiles(str(s))
        if mol: fps.append(list(AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=n_bits)))
        else: fps.append([0]*n_bits)
    return np.array(fps, dtype=np.float32)

def ecfp6_batch(smiles_list, n_bits=2048):
    fps = []
    for s in smiles_list:
        mol = Chem.MolFromSmiles(str(s))
        if mol: fps.append(list(AllChem.GetMorganFingerprintAsBitVect(mol, 3, nBits=n_bits)))
        else: fps.append([0]*n_bits)
    return np.array(fps, dtype=np.float32)

def maccs_batch(smiles_list):
    fps = []
    for s in smiles_list:
        mol = Chem.MolFromSmiles(str(s))
        if mol: fps.append(list(MACCSkeys.GenMACCSKeys(mol)))
        else: fps.append([0]*167)
    return np.array(fps, dtype=np.float32)[:, 1:]  # drop bit 0

def atom_pair_batch(smiles_list, n_bits=2048):
    fps = []
    for s in smiles_list:
        mol = Chem.MolFromSmiles(str(s))
        if mol:
            fp = rdMolDescriptors.GetHashedAtomPairFingerprintAsBitVect(mol, nBits=n_bits)
            fps.append(list(fp))
        else: fps.append([0]*n_bits)
    return np.array(fps, dtype=np.float32)

def topo_torsion_batch(smiles_list, n_bits=2048):
    fps = []
    for s in smiles_list:
        mol = Chem.MolFromSmiles(str(s))
        if mol:
            fp = rdMolDescriptors.GetHashedTopologicalTorsionFingerprintAsBitVect(mol, nBits=n_bits)
            fps.append(list(fp))
        else: fps.append([0]*n_bits)
    return np.array(fps, dtype=np.float32)

def rdkit_fp_batch(smiles_list, n_bits=2048):
    fps = []
    for s in smiles_list:
        mol = Chem.MolFromSmiles(str(s))
        if mol:
            fp = Chem.RDKFingerprint(mol, fpSize=n_bits)
            fps.append(list(fp))
        else: fps.append([0]*n_bits)
    return np.array(fps, dtype=np.float32)

PHYS_PROPS = ["mw","logp","tpsa","hbd","hba","rotbonds","fsp3",
               "n_rings","n_aromatic_rings","heavy_atoms","formal_charge"]

def physchem_batch(smiles_list):
    rows = []
    for s in smiles_list:
        p = compute_physchem(str(s))
        rows.append([p.get(k, 0) or 0 for k in PHYS_PROPS])
    return np.array(rows, dtype=np.float32)

In [4]:
tr = load_train(); te = load_test()
y_tr = tr["pec50"].values.astype(np.float64)
scaffolds = tr["smiles"].map(bemis_murcko).tolist()
splits = scaffold_kfold_indices(scaffolds, N_FOLDS, SEED)

X_tr = impute(combined(tr["smiles"].tolist()))
X_te = impute(combined(te["smiles"].tolist()))

print("Computing ECFP4...", flush=True)
fps4_tr = ecfp4_batch(tr["smiles"].tolist())
fps4_te = ecfp4_batch(te["smiles"].tolist())

print("Computing ECFP6...", flush=True)
fps6_tr = ecfp6_batch(tr["smiles"].tolist())
fps6_te = ecfp6_batch(te["smiles"].tolist())

print("Computing MACCS...", flush=True)
maccs_tr = maccs_batch(tr["smiles"].tolist())
maccs_te = maccs_batch(te["smiles"].tolist())

print("Computing Atom-pair FPs...", flush=True)
ap_tr = atom_pair_batch(tr["smiles"].tolist())
ap_te = atom_pair_batch(te["smiles"].tolist())

print("Computing Topological-Torsion FPs...", flush=True)
tt_tr = topo_torsion_batch(tr["smiles"].tolist())
tt_te = topo_torsion_batch(te["smiles"].tolist())

print("Computing physchem...", flush=True)
phys_tr = physchem_batch(tr["smiles"].tolist())
phys_te = physchem_batch(te["smiles"].tolist())

cliff_pairs = (pd.read_parquet(DATA_PROCESSED/"cliff_pairs.parquet")
               if (DATA_PROCESSED/"cliff_pairs.parquet").exists() else pd.DataFrame())
if len(cliff_pairs) > 0:
    s2i = {s:i for i,s in enumerate(tr["smiles"].tolist())}
    ac = "cliff_active_smiles" if "cliff_active_smiles" in cliff_pairs.columns else "smiles_a"
    ic = "cliff_inactive_smiles" if "cliff_inactive_smiles" in cliff_pairs.columns else "smiles_b"
    cliff_pairs["idx_active"]   = cliff_pairs[ac].map(s2i)
    cliff_pairs["idx_inactive"] = cliff_pairs[ic].map(s2i)
    cliff_pairs = cliff_pairs.dropna(subset=["idx_active","idx_inactive"])
    cliff_pairs[["idx_active","idx_inactive"]] = cliff_pairs[["idx_active","idx_inactive"]].astype(int)
print(f"Train {len(tr):,}  Test {len(te):,}  Cliffs {len(cliff_pairs)}")

Computing ECFP4...


[03:37:00] DEPRECATION WARNING: please use MorganGenerator
[03:37:00] DEPRECATION WARNING: please use MorganGenerator
[03:37:00] DEPRECATION WARNING: please use MorganGenerator
[03:37:00] DEPRECATION WARNING: please use MorganGenerator
[03:37:00] DEPRECATION WARNING: please use MorganGenerator
[03:37:00] DEPRECATION WARNING: please use MorganGenerator
[03:37:00] DEPRECATION WARNING: please use MorganGenerator
[03:37:00] DEPRECATION WARNING: please use MorganGenerator
[03:37:00] DEPRECATION WARNING: please use MorganGenerator
[03:37:01] DEPRECATION WARNING: please use MorganGenerator
[03:37:01] DEPRECATION WARNING: please use MorganGenerator
[03:37:01] DEPRECATION WARNING: please use MorganGenerator
[03:37:01] DEPRECATION WARNING: please use MorganGenerator
[03:37:01] DEPRECATION WARNING: please use MorganGenerator
[03:37:01] DEPRECATION WARNING: please use MorganGenerator
[03:37:01] DEPRECATION WARNING: please use MorganGenerator
[03:37:01] DEPRECATION WARNING: please use MorganGenerat

[03:37:01] DEPRECATION WARNING: please use MorganGenerator
[03:37:01] DEPRECATION WARNING: please use MorganGenerator
[03:37:01] DEPRECATION WARNING: please use MorganGenerator
[03:37:01] DEPRECATION WARNING: please use MorganGenerator
[03:37:01] DEPRECATION WARNING: please use MorganGenerator
[03:37:01] DEPRECATION WARNING: please use MorganGenerator
[03:37:01] DEPRECATION WARNING: please use MorganGenerator
[03:37:01] DEPRECATION WARNING: please use MorganGenerator
[03:37:01] DEPRECATION WARNING: please use MorganGenerator
[03:37:01] DEPRECATION WARNING: please use MorganGenerator
[03:37:01] DEPRECATION WARNING: please use MorganGenerator
[03:37:01] DEPRECATION WARNING: please use MorganGenerator
[03:37:01] DEPRECATION WARNING: please use MorganGenerator
[03:37:01] DEPRECATION WARNING: please use MorganGenerator
[03:37:01] DEPRECATION WARNING: please use MorganGenerator
[03:37:01] DEPRECATION WARNING: please use MorganGenerator
[03:37:01] DEPRECATION WARNING: please use MorganGenerat

[03:37:01] DEPRECATION WARNING: please use MorganGenerator
[03:37:01] DEPRECATION WARNING: please use MorganGenerator
[03:37:01] DEPRECATION WARNING: please use MorganGenerator
[03:37:01] DEPRECATION WARNING: please use MorganGenerator
[03:37:01] DEPRECATION WARNING: please use MorganGenerator
[03:37:01] DEPRECATION WARNING: please use MorganGenerator
[03:37:01] DEPRECATION WARNING: please use MorganGenerator
[03:37:01] DEPRECATION WARNING: please use MorganGenerator
[03:37:01] DEPRECATION WARNING: please use MorganGenerator
[03:37:01] DEPRECATION WARNING: please use MorganGenerator
[03:37:01] DEPRECATION WARNING: please use MorganGenerator
[03:37:01] DEPRECATION WARNING: please use MorganGenerator
[03:37:01] DEPRECATION WARNING: please use MorganGenerator
[03:37:01] DEPRECATION WARNING: please use MorganGenerator
[03:37:01] DEPRECATION WARNING: please use MorganGenerator
[03:37:01] DEPRECATION WARNING: please use MorganGenerator
[03:37:01] DEPRECATION WARNING: please use MorganGenerat

[03:37:01] DEPRECATION WARNING: please use MorganGenerator
[03:37:01] DEPRECATION WARNING: please use MorganGenerator
[03:37:01] DEPRECATION WARNING: please use MorganGenerator
[03:37:01] DEPRECATION WARNING: please use MorganGenerator
[03:37:01] DEPRECATION WARNING: please use MorganGenerator
[03:37:01] DEPRECATION WARNING: please use MorganGenerator
[03:37:01] DEPRECATION WARNING: please use MorganGenerator
[03:37:01] DEPRECATION WARNING: please use MorganGenerator
[03:37:01] DEPRECATION WARNING: please use MorganGenerator
[03:37:01] DEPRECATION WARNING: please use MorganGenerator
[03:37:01] DEPRECATION WARNING: please use MorganGenerator
[03:37:01] DEPRECATION WARNING: please use MorganGenerator
[03:37:01] DEPRECATION WARNING: please use MorganGenerator
[03:37:01] DEPRECATION WARNING: please use MorganGenerator
[03:37:01] DEPRECATION WARNING: please use MorganGenerator
[03:37:01] DEPRECATION WARNING: please use MorganGenerator
[03:37:01] DEPRECATION WARNING: please use MorganGenerat

...

[03:37:21] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[03:37:21] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[03:37:21] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[03:37:21] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[03:37:21] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[03:37:21] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[03:37:21] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[03:37:21] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[03:37:21] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[03:37:21] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[03:37:21] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[03:37:21] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[03:37:21] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[03:37:21] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[03:37

[03:37:21] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[03:37:21] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[03:37:21] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[03:37:21] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[03:37:21] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[03:37:21] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[03:37:21] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[03:37:21] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[03:37:21] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[03:37:21] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[03:37:21] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[03:37:21] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[03:37:21] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[03:37:21] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[03:37

Computing physchem...


[03:37:21] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[03:37:21] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[03:37:21] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[03:37:21] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[03:37:21] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[03:37:21] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[03:37:21] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[03:37:21] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[03:37:21] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[03:37:21] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[03:37:21] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[03:37:21] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[03:37:21] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[03:37:21] DEPRECATION WARNING: please use TopologicalTorsionGenerator
[03:37

Train 4,139  Test 513  Cliffs 0


In [5]:
# --- Tanimoto (ECFP4-based) ---
print("Computing pairwise Tanimoto...", flush=True)
dot_tt = (fps4_tr @ fps4_tr.T).astype(np.float32)
rowsum = fps4_tr.sum(1).astype(np.float32)
union_tt = rowsum[:,None] + rowsum[None,:] - dot_tt
tanimoto_tr = np.where(union_tt>0, dot_tt/union_tt, 0.0)
np.fill_diagonal(tanimoto_tr, 0.0)

dot_te = (fps4_te @ fps4_tr.T).astype(np.float32)
rs_te = fps4_te.sum(1)[:,None]; rs_tr_v = fps4_tr.sum(1)[None,:]
sim_te_tr = dot_te / np.maximum(rs_te + rs_tr_v - dot_te, 1e-6)

Computing pairwise Tanimoto...


In [6]:
# --- All-fingerprint delta features ---
def compress_fp(fp, out_dim=64):
    N, D = fp.shape; block = D // out_dim
    return fp[:, :block*out_dim].reshape(N, out_dim, block).mean(-1).astype(np.float32)

def compress_maccs(fp, out_dim=32):
    N, D = fp.shape; block = D // out_dim
    return fp[:, :block*out_dim].reshape(N, out_dim, block).mean(-1).astype(np.float32)

def make_allfp_delta_feats(fp4_a, fp4_q, fp6_a, fp6_q, maccs_a, maccs_q,
                            ap_a, ap_q, tt_a, tt_q,
                            sim_col, anchor_pec50, phys_diff):
    def common_diff(fa, fb, outdim=64):
        c = np.minimum(fa, fb).astype(np.float32)
        d = np.abs(fa - fb).astype(np.float32)
        return compress_fp(c, outdim), compress_fp(d, outdim)

    c4, d4 = common_diff(fp4_a, fp4_q)   # 64+64
    c6, d6 = common_diff(fp6_a, fp6_q)   # 64+64
    cap, dap = common_diff(ap_a, ap_q)   # 64+64
    ctt, dtt = common_diff(tt_a, tt_q)   # 64+64

    maccs_d = np.abs(maccs_a - maccs_q).astype(np.float32)
    cm = compress_maccs(maccs_d, 32)     # 32

    # Total: 64*8 + 32 + 1 + 1 + 11 = 556 features
    return np.hstack([c4, d4, c6, d6, cap, dap, ctt, dtt, cm,
                      sim_col, anchor_pec50[:,None], phys_diff])

# Verify shape
_test = make_allfp_delta_feats(
    fps4_tr[:2], fps4_tr[2:4], fps6_tr[:2], fps6_tr[2:4],
    maccs_tr[:2], maccs_tr[2:4], ap_tr[:2], ap_tr[2:4],
    tt_tr[:2], tt_tr[2:4],
    np.ones((2,1)), y_tr[:2], phys_tr[:2]-phys_tr[2:4]
)
print(f"All-FP delta feature dim: {_test.shape[1]}")

All-FP delta feature dim: 557


In [7]:
# --- 3-Tier training pairs ---
TIERS = {"HIGH": (0.60, 0.90), "MED": (0.45, 0.60), "LOW": (0.35, 0.45)}
i_idx_gl, j_idx_gl = np.where(np.triu(tanimoto_tr > 0.30, k=1))
sim_gl = tanimoto_tr[i_idx_gl, j_idx_gl]

def build_all_tier(tier_name, sim_lo, sim_hi):
    if tier_name == "HIGH": mask = (sim_gl >= sim_lo) & (sim_gl <= sim_hi)
    else: mask = (sim_gl >= sim_lo) & (sim_gl < sim_hi)
    ii, jj = i_idx_gl[mask], j_idx_gl[mask]
    sim_ij = sim_gl[mask][:,None]
    pd_ij = phys_tr[jj] - phys_tr[ii]
    F_ij = make_allfp_delta_feats(
        fps4_tr[ii], fps4_tr[jj], fps6_tr[ii], fps6_tr[jj],
        maccs_tr[ii], maccs_tr[jj], ap_tr[ii], ap_tr[jj],
        tt_tr[ii], tt_tr[jj], sim_ij, y_tr[ii], pd_ij)
    F_ji = make_allfp_delta_feats(
        fps4_tr[jj], fps4_tr[ii], fps6_tr[jj], fps6_tr[ii],
        maccs_tr[jj], maccs_tr[ii], ap_tr[jj], ap_tr[ii],
        tt_tr[jj], tt_tr[ii], sim_ij, y_tr[jj], -pd_ij)
    F = np.vstack([F_ij, F_ji])
    y = np.concatenate([y_tr[jj]-y_tr[ii], y_tr[ii]-y_tr[jj]])
    return F, y, len(ii)

for tier, (lo, hi) in TIERS.items():
    _, _, n = build_all_tier(tier, lo, hi)
    print(f"  {tier} [{lo},{hi}]: {n:,} pairs")

  HIGH [0.6,0.9]: 117 pairs


  MED [0.45,0.6]: 910 pairs


  LOW [0.35,0.45]: 4,150 pairs


In [8]:
# --- Train tier delta models ---
DELTA_LGBM = dict(n_estimators=1200, num_leaves=63, learning_rate=0.04,
                  min_child_samples=15, subsample=0.8, colsample_bytree=0.7,
                  reg_alpha=0.05, reg_lambda=0.1, random_state=SEED, verbose=-1, n_jobs=4)

tier_models = {}
for tier, (lo, hi) in TIERS.items():
    F, y, n = build_all_tier(tier, lo, hi)
    print(f"Training {tier} on {len(F):,} pairs...", flush=True)
    m = lgb.LGBMRegressor(**DELTA_LGBM)
    m.fit(F, y, callbacks=[lgb.log_evaluation(-1)])
    tier_models[tier] = m
    print(f"  {tier} done.", flush=True)

Training HIGH on 234 pairs...


  HIGH done.


Training MED on 1,820 pairs...


  MED done.


Training LOW on 8,300 pairs...


  LOW done.


In [9]:
K_NEIGHBORS = 10

def predict_allfp(fps4_q, fps4_r, fps6_q, fps6_r, maccs_q, maccs_r,
                  ap_q, ap_r, tt_q, tt_r,
                  y_ref, phys_q, phys_r, sim_matrix, fallback_preds):
    N = len(fps4_q)
    preds = np.full(N, np.nan)
    tier_counts = {t: 0 for t in TIERS}; tier_counts["fallback"] = 0

    for qi in range(N):
        sim_row = sim_matrix[qi]
        assigned = False
        for tier, (lo, hi) in TIERS.items():
            if tier == "HIGH": cand_mask = (sim_row >= lo) & (sim_row <= hi)
            else: cand_mask = (sim_row >= lo) & (sim_row < hi)
            cand_idx = np.where(cand_mask)[0]
            if len(cand_idx) == 0: continue

            top_k = np.argsort(-sim_row[cand_idx])[:K_NEIGHBORS]
            sel_idx = cand_idx[top_k]
            cand_sims = sim_row[sel_idx]

            fp4q = np.tile(fps4_q[qi:qi+1], (len(sel_idx),1))
            fp6q = np.tile(fps6_q[qi:qi+1], (len(sel_idx),1))
            macq = np.tile(maccs_q[qi:qi+1], (len(sel_idx),1))
            apq  = np.tile(ap_q[qi:qi+1],   (len(sel_idx),1))
            ttq  = np.tile(tt_q[qi:qi+1],   (len(sel_idx),1))
            pd   = phys_q[qi:qi+1] - phys_r[sel_idx]

            F_k = make_allfp_delta_feats(
                fps4_r[sel_idx], fp4q, fps6_r[sel_idx], fp6q,
                maccs_r[sel_idx], macq, ap_r[sel_idx], apq,
                tt_r[sel_idx], ttq,
                cand_sims[:,None], y_ref[sel_idx], pd)
            delta_k = tier_models[tier].predict(F_k)
            template_preds = y_ref[sel_idx] + delta_k
            weights = cand_sims ** 2
            preds[qi] = np.average(template_preds, weights=weights)
            tier_counts[tier] += 1
            assigned = True
            break

        if not assigned:
            preds[qi] = fallback_preds[qi]
            tier_counts["fallback"] += 1

    return preds, tier_counts

print("All-FP predict function ready.")

All-FP predict function ready.


In [10]:
# --- Scaffold 5-fold CV ---
print("\n=== Scaffold 5-fold CV ===", flush=True)
oof_delta = np.full(len(y_tr), np.nan)
oof_direct = np.full(len(y_tr), np.nan)

for fold, (tr_idx, va_idx) in enumerate(splits):
    m_dir = lgb.train(LGBM_BASE, lgb.Dataset(X_tr[tr_idx], label=y_tr[tr_idx]),
                      valid_sets=[lgb.Dataset(X_tr[va_idx], label=y_tr[va_idx])],
                      callbacks=[lgb.early_stopping(60,verbose=False), lgb.log_evaluation(-1)])
    oof_direct[va_idx] = m_dir.predict(X_tr[va_idx])

    fps4_va = fps4_tr[va_idx]; fps4_ft = fps4_tr[tr_idx]
    dot_vf = (fps4_va @ fps4_ft.T).astype(np.float32)
    rs_v = fps4_va.sum(1)[:,None]; rs_f = fps4_ft.sum(1)[None,:]
    sim_vf = dot_vf / np.maximum(rs_v + rs_f - dot_vf, 1e-6)

    preds_d, tc = predict_allfp(
        fps4_va, fps4_ft, fps6_tr[va_idx], fps6_tr[tr_idx],
        maccs_tr[va_idx], maccs_tr[tr_idx],
        ap_tr[va_idx], ap_tr[tr_idx], tt_tr[va_idx], tt_tr[tr_idx],
        y_tr[tr_idx], phys_tr[va_idx], phys_tr[tr_idx],
        sim_vf, oof_direct[va_idx])
    oof_delta[va_idx] = preds_d

    r_dir = rae(y_tr[va_idx], oof_direct[va_idx])
    r_dlt = rae(y_tr[va_idx], oof_delta[va_idx])
    print(f"  fold {fold+1}  direct={r_dir:.4f}  allfp_delta={r_dlt:.4f}  tiers={tc}", flush=True)

m_dir = full_metrics(y_tr, oof_direct, "direct_lgbm")
m_dlt = full_metrics(y_tr, oof_delta,  "allfp_delta_3tier")

# Compare vs nb111
nb111_path = DATA_PROCESSED / "oof_enhanced_delta_3tier.npy"
if nb111_path.exists():
    r111 = rae(y_tr, np.load(nb111_path))
    print(f"\nnb111 (ECFP4+ECFP6+MACCS):  {r111:.4f}")
    print(f"nb117 (all fingerprints):    {m_dlt['RAE']:.4f}")
    print(f"Delta: {m_dlt['RAE'] - r111:+.4f}")


=== Scaffold 5-fold CV ===


  fold 1  direct=0.4920  allfp_delta=0.2053  tiers={'HIGH': 12, 'MED': 177, 'LOW': 344, 'fallback': 295}


  fold 2  direct=0.5734  allfp_delta=0.2253  tiers={'HIGH': 15, 'MED': 170, 'LOW': 332, 'fallback': 311}


  fold 3  direct=0.5979  allfp_delta=0.2619  tiers={'HIGH': 18, 'MED': 151, 'LOW': 323, 'fallback': 336}


  fold 4  direct=0.5647  allfp_delta=0.2406  tiers={'HIGH': 16, 'MED': 164, 'LOW': 329, 'fallback': 319}


  fold 5  direct=0.5954  allfp_delta=0.2436  tiers={'HIGH': 15, 'MED': 161, 'LOW': 328, 'fallback': 323}


  [direct_lgbm] RAE=0.5598 MAE=0.5093 R2=0.6075 r=0.7794 rho=0.7337
  [allfp_delta_3tier] RAE=0.2333 MAE=0.2122 R2=0.8407 r=0.9169 rho=0.8919

nb111 (ECFP4+ECFP6+MACCS):  0.2480
nb117 (all fingerprints):    0.2333
Delta: -0.0147


In [11]:
# --- Final test predictions ---
print("\nFitting final direct LGBM...", flush=True)
m_final = lgb.train(LGBM_BASE, lgb.Dataset(X_tr, label=y_tr), callbacks=[lgb.log_evaluation(-1)])
te_direct = m_final.predict(X_te)

print("Running all-FP delta on test...", flush=True)
te_delta, te_tc = predict_allfp(
    fps4_te, fps4_tr, fps6_te, fps6_tr,
    maccs_te, maccs_tr, ap_te, ap_tr, tt_te, tt_tr,
    y_tr, phys_te, phys_tr,
    sim_te_tr, te_direct)
print(f"Test tier usage: {te_tc}")

te_preds = np.clip(te_delta, y_tr.min()-0.5, y_tr.max()+0.5)

np.save(DATA_PROCESSED/"oof_allfp_delta_3tier.npy", oof_delta)
np.save(DATA_PROCESSED/"te_oof_allfp_delta_3tier.npy", te_preds)
sub = pd.DataFrame({"Molecule Name": te["name"].values, "pEC50": te_preds})
assert len(sub)==513 and sub["pEC50"].notna().all()
p = SUBMISSIONS/"117_delta_all_fingerprints.csv"; sub.to_csv(p, index=False)
print(f"Saved {p}")
print(f"Test: min={te_preds.min():.2f} med={np.median(te_preds):.2f} max={te_preds.max():.2f}")
print(f"\n*** nb117 OOF RAE = {m_dlt['RAE']:.4f} ***")


Fitting final direct LGBM...


Running all-FP delta on test...


Test tier usage: {'HIGH': 91, 'MED': 369, 'LOW': 50, 'fallback': 3}
Saved D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\submissions\117_delta_all_fingerprints.csv
Test: min=3.29 med=4.95 max=6.74

*** nb117 OOF RAE = 0.2333 ***
